# EnerGIS Framework - RunnerHaupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.## ÜbersichtDieses Notebook führt einen vollständigen Optimierungslauf durch:- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont- **Model Predictive Control (MPC)**: RH mit Forecast-Updates- **PF → RH/MPC**: Kombinierter Workflow mit Design-Fixierung## Quick Start1. Alle Zellen mit **Run All** ausführen2. Bei Bedarf Config-Pfade in Zelle 2 anpassen3. Ergebnisse werden in `exports/` gespeichert## ExportDer Runner exportiert automatisch:- 📊 **Visualisierungen**: Wärmebilanz, elektrische Bilanz, Speicher, Kostenaufschlüsselung- 📈 **CSV Dateien**: Zeitreihen für weitere Analysen- 📋 **JSON Dateien**: Kosten, Design, Zusammenfassung- 📦 **Excel Bundle** (optional in Zelle 13): Vollständige Ergebnisse mit allen ZeitreihenAlle Exports werden in `notebooks/exports/latest_run/` gespeichert.---

## 1. Setup & Imports

In [ ]:
# Auto-Setup: Projekt-Root finden und zum Path hinzufügen
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    """Findet das Projekt-Root-Verzeichnis."""
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

In [ ]:
# Imports
import warnings
from datetime import datetime

from energis.run import rolling_horizon as rh
from energis.run import orchestrator

warnings.filterwarnings('ignore')
print("✅ Imports erfolgreich")

## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Standard-Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt, etc.)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone, etc.)
- `baseline.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `pf_then_rh.workflow.scenario.yaml` - Szenario (Run-Mode, RH-Parameter)

Passe die Config-Pfade nach Bedarf an!

In [ ]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# Config-Dateien prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

## 3. Workflow ausführenDer Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:- **PF_ONLY**: Nur Perfect Forecast- **RH_ONLY**: Nur Rolling Horizon- **MPC_ONLY**: Nur Model Predictive Control (mit Forecasts)- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design- **PF_THEN_MPC**: PF für Dimensionierung, dann MPC mit fixiertem Design

In [ ]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

## 4. Ergebnisse

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [ ]:
if optimization_success and workflow:
    print("\n" + "="*70)
    print("📊 ERGEBNISSE")
    print("="*70)
    
    # Perfect Forecast Ergebnisse
    if workflow.pf_result:
        print("\n🎯 Perfect Forecast (PF):")
        print(f"  Zeitschritte:  {len(workflow.pf_result.table)}")
        
        if workflow.pf_result.costs:
            obj_value = workflow.pf_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
            
            # Detailed cost breakdown
            print("\n  💰 Detaillierte Kostenaufschlüsselung:")
            
            # Electricity costs
            elec_total = workflow.pf_result.costs.get('objective.Grid_energy_cost_EUR', 0.0)
            elec_base = workflow.pf_result.costs.get('objective.Electricity_base_cost_EUR', 0.0)
            elec_fee = workflow.pf_result.costs.get('objective.Electricity_energy_fee_EUR', 0.0)
            elec_grid = workflow.pf_result.costs.get('objective.Electricity_grid_fee_EUR', 0.0)
            demand_charge = workflow.pf_result.costs.get('objective.Demand_charge_cost_EUR', 0.0)
            
            if elec_total > 1e-3:
                print(f"    Stromkosten:           {elec_total:>12,.0f} EUR")
                if elec_base > 1e-3:
                    print(f"      - Spotpreis:         {elec_base:>12,.0f} EUR")
                if elec_fee > 1e-3:
                    print(f"      - Energy Fee:        {elec_fee:>12,.0f} EUR")
                if elec_grid > 1e-3:
                    print(f"      - Netzentgelte:      {elec_grid:>12,.0f} EUR")
                if demand_charge > 1e-3:
                    print(f"      - Leistungspreis:    {demand_charge:>12,.0f} EUR")
            
            # Fuel costs
            fuel_total = workflow.pf_result.costs.get('objective.Fuel_cost_EUR', 0.0)
            if fuel_total > 1e-3:
                print(f"    Brennstoffkosten:      {fuel_total:>12,.0f} EUR")
                # Show breakdown by fuel type
                for key, value in workflow.pf_result.costs.items():
                    if key.startswith('objective.Fuel_cost_') and key != 'objective.Fuel_cost_EUR' and key.endswith('_EUR'):
                        fuel_type = key.replace('objective.Fuel_cost_', '').replace('_EUR', '').title()
                        if value > 1e-3:
                            print(f"      - {fuel_type:15s}:  {value:>12,.0f} EUR")
            
            # Investment costs
            capex_total = workflow.pf_result.costs.get('objective.Capex_cost_EUR', 0.0)
            capex_hp = workflow.pf_result.costs.get('objective.Capex_heat_pumps_EUR', 0.0)
            capex_sto = workflow.pf_result.costs.get('objective.Capex_storage_EUR', 0.0)
            
            if capex_total > 1e-3:
                print(f"    Investitionskosten:    {capex_total:>12,.0f} EUR")
                if capex_hp > 1e-3:
                    print(f"      - Wärmepumpen:       {capex_hp:>12,.0f} EUR")
                if capex_sto > 1e-3:
                    print(f"      - Speicher:          {capex_sto:>12,.0f} EUR")
            
            # Other costs
            co2_cost = workflow.pf_result.costs.get('objective.CO2_cost_EUR', 0.0)
            dump_cost = workflow.pf_result.costs.get('objective.Dump_cost_EUR', 0.0)
            
            if co2_cost > 1e-3:
                print(f"    CO2 Kosten:            {co2_cost:>12,.0f} EUR")
            if dump_cost > 1e-3:
                print(f"    Abfall/Dump:           {dump_cost:>12,.0f} EUR")
            
            # Revenue
            revenue = workflow.pf_result.costs.get('objective.Grid_sell_revenue_EUR', 0.0)
            if revenue > 1e-3:
                print(f"    Erlöse (Einspeisung):  {revenue:>12,.0f} EUR")
            
            peak_power = workflow.pf_result.costs.get('objective.P_buy_peak_MW')
            if peak_power is not None:
                print(f"\n  Peak-Leistung:         {peak_power:.2f} MW")
    
    # Rolling Horizon Ergebnisse
    if workflow.rh_result:
        print("\n🔄 Rolling Horizon (RH):")
        print(f"  Fenster:       {len(workflow.rh_result.windows)}")
        print(f"  Zeitschritte:  {len(workflow.rh_result.table)}")
        
        if workflow.rh_result.costs:
            obj_value = workflow.rh_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
    
    # Model Predictive Control Ergebnisse
    if workflow.mpc_result:
        print("\n🔮 Model Predictive Control (MPC):")
        print(f"  Fenster:       {len(workflow.mpc_result.windows)}")
        print(f"  Zeitschritte:  {len(workflow.mpc_result.table)}")
        
        if workflow.mpc_result.costs:
            obj_value = workflow.mpc_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
            
            # Show forecast method if available
            forecast_method = workflow.config.get('scenario', {}).get('mpc', {}).get('forecast_method', 'unknown')
            print(f"  Forecast:      {forecast_method}")
    
    # Design
    if workflow.design:
        print("\n🏭 Anlagen-Design:")
        
        if workflow.design.heat_pumps:
            print("  Wärmepumpen:")
            for hp_id, hp_data in sorted(workflow.design.heat_pumps.items()):
                capacity = hp_data.get('capacity_mw', 0.0)
                print(f"    {hp_id}: {capacity:.2f} MW")
        
        if workflow.design.storage:
            storage_capacity = workflow.design.storage.get('capacity_mwh', 0.0)
            print(f"  Speicher:      {storage_capacity:.2f} MWh")
        
        if workflow.design.generators:
            print("  Generatoren:")
            for gen_id, gen_data in sorted(workflow.design.generators.items()):
                capacity = gen_data.get('capacity_mw', 0.0)
                print(f"    {gen_id}: {capacity:.2f} MW")
    
    print("\n" + "="*70)
else:
    print("⚠️  Keine Ergebnisse verfügbar")

## 5. Export (Optional)

Vollständiger Export aller Ergebnisse als Excel/CSV/JSON in `exports/`.

In [ ]:
# Export aktiviert - Vollständige Ergebnisse exportieren
if optimization_success:
    print("📦 Exportiere Ergebnisse...")
    export_meta = orchestrator.run_all(CONFIG_PATHS, overrides=OVERRIDES)
    
    print(f"\n✅ Export abgeschlossen:")
    print(f"  Verzeichnis: {export_meta['outdir']}")
    print(f"  Excel:       {export_meta.get('scenario_xlsx')}")
    print(f"  Design-JSON: {export_meta.get('pf_design_json')}")
    
    # Plots wurden automatisch erstellt
    plots = export_meta.get('plots', [])
    if plots:
        print(f"\n📊 Visualisierungen ({len(plots)} Dateien):")
        for plot_path in plots[:5]:  # Erste 5 anzeigen
            print(f"    - {plot_path}")
        if len(plots) > 5:
            print(f"    ... und {len(plots) - 5} weitere")
else:
    print("⚠️  Export übersprungen (Optimierung fehlgeschlagen)")

## 6. Detaillierte Analyse (Optional)

Für detaillierte Analysen und Visualisierungen siehe:
- `scenario_studio.ipynb` - Interaktives Dashboard mit Plots und KPIs
- `synthetic_example.ipynb` - Beispiel mit synthetischen Daten
- `validation.ipynb` - Validierung gegen Referenz-Daten

---

## Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`